In [1]:
!pip install polars

In [2]:
from sklearn.multioutput import MultiOutputClassifier
import xgboost as xgb

import gc
import numpy as np
import polars as pl
import json
import pandas as pd

from sklearn.metrics import roc_auc_score

In [3]:
try:
    import pyarrow
    print("PyArrow is installed. Version:", pyarrow.__version__)
except ModuleNotFoundError:
    print("PyArrow is NOT found in this environment. You installed it in a different one.")

PyArrow is installed. Version: 23.0.1


In [4]:
target = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/train_target.parquet')

In [5]:
train = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-1600/train_combined_1600.parquet')

In [6]:
test = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-1600/test_combined_1600.parquet')

In [7]:
cat_feature_names = [
    col_name for col_name in train.columns
    if col_name.startswith("cat_feature")
]

In [8]:
X_train_pd = train.drop('customer_id').to_pandas().astype('float32')

In [9]:
y_train_pd = target.drop('customer_id').to_pandas()

In [10]:
xgb_base = xgb.XGBClassifier(
    n_estimators=500,               
    max_depth=6,   
    learning_rate=0.05,             
    
    tree_method='hist',          
    device='cuda',              
    max_bin=128,
    n_jobs=1,
           
    random_state=42,
    verbosity=1,                    
    missing=np.nan,                 
    objective='binary:logistic',   
    eval_metric='logloss'          
)

multi_xgb = MultiOutputClassifier(xgb_base, n_jobs=1)
multi_xgb.fit(X_train_pd, y_train_pd)

MultiOutputClassifier(estimator=XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device='cuda',
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='logloss',
                                              feature_types=None,
                                              feature_weights=None, gamma=None,
                                              grow_policy=None,
                                              importance_type=None,
                                              interaction_constraints=None,
                                              learning_rate=0.05, max_bin=128,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=6,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=500, n_jobs=1,
                                              num_parallel_tree=None, ...),
                      n_jobs=1)

In [11]:
sample_submit = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/sample_submit.parquet')
raw_preds = []

for est in multi_xgb.estimators_:
    # Преобразуем тестовые данные (удаляем customer_id, переводим в float32)
    X_test = test.drop('customer_id').to_numpy().astype('float32')
    # Получаем логиты напрямую
    logits = est.predict(X_test, output_margin=True)
    raw_preds.append(logits)

test_raw = np.column_stack(raw_preds)

test_raw = test_raw.astype(np.float64)

predict_schema = [col.replace("target_", "predict_") for col in target.columns if col.startswith("target_")]

hgb_submit = pl.DataFrame(
    test_raw,
    schema=predict_schema,
    orient='row'
)

hgb_submit = pl.concat([test.select('customer_id'), hgb_submit], how='horizontal')

hgb_submit.write_parquet("xgb_submit_1600.parquet")

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [09:37:06] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
